# Module 17: Embodied Active Inference
## From Grid Worlds to Physical Bodies

**Spinning Up in Active Inference** | Notebook 17

---

In Module 1 we met Thorndike's cats, struggling against puzzle-box latches through trial and error. Every module since has worked in a world of discrete grids: north, south, east, west. These abstractions let us build intuition for value functions, prediction errors, generative models, and the Expected Free Energy -- but they leave out something fundamental.

Real organisms do not move between grid cells. They have *bodies*: joints that bend, muscles that pull, skin that senses pressure. A cat escaping a puzzle box must coordinate dozens of degrees of freedom -- paws, claws, shoulders, spine -- all under noisy, delayed proprioceptive feedback. The intelligence is not just in the brain; it is distributed across the sensorimotor loop.

This insight has a long history:
- **Braitenberg (1984)** showed that vehicles with two sensors and two motors can exhibit fear, aggression, love, and exploration -- complex behavior from simple embodied wiring.
- **Brooks (1986)** argued that "the world is its own best model" -- subsumption architectures that react to the physical world outperformed classical planners.
- **Pfeifer & Bongard (2006)** demonstrated that body morphology *shapes* the computational problems the brain must solve.
- **Friston (2010)** unified all of this: the Free Energy Principle was always about *embodied* organisms minimizing surprise through action on their environment.

In this capstone module we close the loop. We build a simulated creature with joints, link lengths, and torques -- a 2-link planar arm in a physics engine -- and show that Active Inference naturally extends from discrete grids to continuous, embodied control. The same principle (minimize free energy) that drove our T-maze agent now drives a reaching arm to its target.

By the end of this module you will:
1. Understand why embodiment matters for intelligence and for AIF
2. Implement a 2-link arm in PyBullet (headless physics simulation)
3. Build a PD controller, an RL controller, and an AIF controller for reaching
4. See how AIF unifies perception and action through prediction error minimization
5. Test robustness: target perturbation, observation noise, morphology change

**Prerequisites:** Modules 1-16 (the entire curriculum).
**Time:** ~75 minutes.

## 17.1 From Discrete to Continuous: What Changes?

In Modules 1-16 our agents lived in grid worlds:

| Concept | Discrete (Modules 1-16) | Continuous (This Module) |
|---------|------------------------|-------------------------|
| **State** | Grid cell index $s \in \{0, 1, \dots, N\}$ | Joint angles $\boldsymbol{\theta} \in \mathbb{R}^n$ |
| **Action** | Cardinal direction $a \in \{\text{N, S, E, W}\}$ | Joint torques $\boldsymbol{\tau} \in \mathbb{R}^n$ |
| **Observation** | Discrete sensory signal | Noisy proprioception $\boldsymbol{o} = \boldsymbol{\theta} + \boldsymbol{\epsilon}$ |
| **Transition B** | $P(s' \mid s, a)$ matrix | Forward dynamics $\boldsymbol{\theta}_{t+1} = f(\boldsymbol{\theta}_t, \boldsymbol{\tau}_t)$ |
| **Observation A** | $P(o \mid s)$ likelihood | Forward kinematics $\boldsymbol{x}_\text{ee} = g(\boldsymbol{\theta})$ |
| **Preferences C** | Log-probability over observations | Target position $\boldsymbol{x}^*$ |
| **Free energy** | $F = \sum_s q(s) [\ln q(s) - \ln P(o,s)]$ | $F = \frac{1}{2} \pi_z \|\boldsymbol{o} - \boldsymbol{\mu}\|^2 + \frac{1}{2} \pi_w \|\boldsymbol{x}^* - g(\boldsymbol{\mu})\|^2$ |

The fundamental insight of Active Inference remains identical: **minimize free energy by updating beliefs (perception) and by acting on the world (action)**. What changes is the math -- sums become integrals, probability tables become differential equations, and softmax policies become gradient-based torque commands.

## 17.2 Mathematical Foundations: Continuous-State Active Inference

### The Generative Model

For a 2-link planar arm with joint angles $\boldsymbol{\theta} = (\theta_1, \theta_2)$ and link lengths $L_1, L_2$:

**Forward kinematics** (the observation model $g$):
$$x_\text{ee} = L_1 \cos\theta_1 + L_2 \cos(\theta_1 + \theta_2)$$
$$y_\text{ee} = L_1 \sin\theta_1 + L_2 \sin(\theta_1 + \theta_2)$$

**Jacobian** (how small changes in $\boldsymbol{\theta}$ map to changes in end-effector position):
$$\mathbf{J}(\boldsymbol{\theta}) = \begin{bmatrix} -L_1 s_1 - L_2 s_{12} & -L_2 s_{12} \\ L_1 c_1 + L_2 c_{12} & L_2 c_{12} \end{bmatrix}$$

where $s_1 = \sin\theta_1$, $c_1 = \cos\theta_1$, $s_{12} = \sin(\theta_1 + \theta_2)$, $c_{12} = \cos(\theta_1 + \theta_2)$.

### Free Energy for Embodied AIF

The variational free energy has two terms:

$$F(\boldsymbol{\mu}) = \underbrace{\frac{1}{2} \pi_z \|\boldsymbol{o} - \boldsymbol{\mu}\|^2}_{\text{proprioceptive prediction error}} + \underbrace{\frac{1}{2} \pi_w \|\boldsymbol{x}^* - g(\boldsymbol{\mu})\|^2}_{\text{sensory/goal prediction error}}$$

- $\boldsymbol{\mu}$: believed joint angles (the brain's best guess)
- $\boldsymbol{o}$: observed joint angles (noisy proprioception)
- $\boldsymbol{x}^*$: preferred end-effector position (the goal)
- $\pi_z$: proprioceptive precision (how much to trust sensors)
- $\pi_w$: goal precision (how strongly to pursue the target)

### Perception: Belief Update

Beliefs are updated by gradient descent on $F$:

$$\dot{\boldsymbol{\mu}} = -\frac{\partial F}{\partial \boldsymbol{\mu}} = \pi_z (\boldsymbol{o} - \boldsymbol{\mu}) + \mathbf{J}(\boldsymbol{\mu})^\top \pi_w (\boldsymbol{x}^* - g(\boldsymbol{\mu}))$$

This is the continuous analog of the Bayesian belief update in discrete AIF: the agent combines sensory evidence (proprioception) with its generative model (kinematics + preferences).

### Action: Making Predictions Come True

In Active Inference, action does not maximize reward -- it *fulfills predictions*. The agent believes its end-effector should be at $\boldsymbol{x}^*$, so it acts to make this belief true:

$$\boldsymbol{\tau} = \mathbf{J}(\boldsymbol{\mu})^\top \cdot \kappa_p \cdot (\boldsymbol{x}^* - g(\boldsymbol{\mu})) - \kappa_d \cdot \dot{\boldsymbol{\theta}}$$

The Jacobian transpose maps task-space prediction errors back to joint-space torques. The damping term $\kappa_d \dot{\boldsymbol{\theta}}$ prevents oscillation. This is formally equivalent to a PD controller, but derived from first principles of free energy minimization rather than hand-tuned control theory.

In [ ]:
# -- Setup -----------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import pybullet as p
import pybullet_data
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import sys
sys.path.insert(0, '..')
from utils.plotting import COLORS, dual_perspective_box

np.random.seed(42)

# ── Arm parameters (used throughout the notebook) ─────────────────────────
L1, L2 = 0.5, 0.4          # link lengths (metres)
LINK_MASSES = [1.0, 0.5]   # link masses (kg)

print("Module 17: Embodied Active Inference")
print(f"  Arm: 2-link planar, L1={L1}m, L2={L2}m")
print(f"  Max reach: {L1 + L2}m")
print(f"  Physics: PyBullet (DIRECT mode, headless)")
print(f"  Visualization: matplotlib")

## 17.3 A Minimal Embodied Agent in PyBullet

[PyBullet](https://pybullet.org/) is an open-source physics engine used widely in robotics research. We run it in `DIRECT` mode (no GUI), which makes it notebook- and Colab-compatible. The arm is a 2-link planar manipulator: two rigid segments connected by revolute joints, rotating in the XY plane.

We define three core functions that encode the arm's **generative model** -- the same role played by the A and B matrices in discrete AIF:
- `forward_kinematics`: maps joint angles to end-effector position (the **observation model**)
- `jacobian`: the derivative of FK, mapping joint-space changes to task-space changes
- `physics_step`: integrates the arm dynamics forward in time (the **transition model**)

In [ ]:
# -- The arm's generative model: kinematics and dynamics --------------------

def forward_kinematics(theta, l1=L1, l2=L2):
    """Observation model g: joint angles -> end-effector (x, y).

    This is the continuous analog of the A-matrix (likelihood mapping)
    from discrete AIF: it maps hidden states to expected observations.
    """
    t1, t2 = theta[0], theta[1]
    x = l1 * np.cos(t1) + l2 * np.cos(t1 + t2)
    y = l1 * np.sin(t1) + l2 * np.sin(t1 + t2)
    return np.array([x, y])


def jacobian(theta, l1=L1, l2=L2):
    """Jacobian J = dg/dtheta: how joint changes map to EE changes.

    This 2x2 matrix is the derivative of forward_kinematics.
    It plays a crucial role in AIF: prediction errors in task space
    are projected back to joint space via J^T.
    """
    t1, t2 = theta[0], theta[1]
    s1, c1 = np.sin(t1), np.cos(t1)
    s12, c12 = np.sin(t1 + t2), np.cos(t1 + t2)
    return np.array([
        [-l1 * s1 - l2 * s12, -l2 * s12],
        [ l1 * c1 + l2 * c12,  l2 * c12],
    ])


def inverse_kinematics(target_xy, l1=L1, l2=L2):
    """Analytic IK for 2-link planar arm (elbow-up solution)."""
    x, y = target_xy
    d = np.clip((x**2 + y**2 - l1**2 - l2**2) / (2 * l1 * l2), -1, 1)
    t2 = np.arctan2(np.sqrt(1 - d**2), d)
    t1 = np.arctan2(y, x) - np.arctan2(l2 * np.sin(t2), l1 + l2 * np.cos(t2))
    return np.array([t1, t2])


def physics_step(theta, dtheta, torque, dt=0.01):
    """Transition model f: one step of simplified arm dynamics.

    Uses decoupled joint dynamics with inertia and damping:
      I * ddtheta = torque - b * dtheta

    This is the continuous analog of the B-matrix (transition mapping).
    """
    I = np.array([0.08, 0.02])   # moments of inertia
    b = 1.5                       # joint damping
    torque = np.clip(torque, -5.0, 5.0)  # torque limits
    ddtheta = (torque - b * dtheta) / I
    dtheta_new = dtheta + ddtheta * dt
    dtheta_new = np.clip(dtheta_new, -10.0, 10.0)  # velocity limits
    theta_new = theta + dtheta_new * dt
    return theta_new, dtheta_new


# ── Verify: FK matches analytic expectations ──────────────────────────────
test_cases = [
    (np.array([0.0, 0.0]),       np.array([L1 + L2, 0.0])),
    (np.array([np.pi/2, 0.0]),   np.array([0.0, L1 + L2])),
    (np.array([np.pi/4, np.pi/4]), None),  # just compute
]

print("Forward kinematics verification:")
for theta_test, expected in test_cases:
    ee = forward_kinematics(theta_test)
    if expected is not None:
        ok = np.allclose(ee, expected, atol=1e-6)
        print(f"  theta={theta_test.round(3)} -> EE=({ee[0]:.4f}, {ee[1]:.4f})  "
              f"expected=({expected[0]:.4f}, {expected[1]:.4f})  {'PASS' if ok else 'FAIL'}")
    else:
        print(f"  theta={theta_test.round(3)} -> EE=({ee[0]:.4f}, {ee[1]:.4f})")

# Verify IK round-trip
target = np.array([0.3, 0.6])
theta_ik = inverse_kinematics(target)
ee_ik = forward_kinematics(theta_ik)
print(f"\nIK round-trip: target={target} -> theta={theta_ik.round(3)} -> EE={ee_ik.round(4)}")
print(f"  Error: {np.linalg.norm(target - ee_ik):.6f}")

In [ ]:
# -- Create the arm in PyBullet and verify physics --------------------------

def create_pybullet_arm(l1=L1, l2=L2):
    """Create a 2-link planar arm in PyBullet (DIRECT mode).

    Returns the physics client ID and arm body ID.
    The arm rotates in the XY plane with revolute joints around Z.
    """
    client = p.connect(p.DIRECT)
    p.setGravity(0, 0, 0)          # no gravity (planar problem)
    p.setTimeStep(1.0 / 240.0)

    arm_id = p.createMultiBody(
        baseMass=0,                 # fixed base
        baseCollisionShapeIndex=-1,
        baseVisualShapeIndex=-1,
        basePosition=[0, 0, 0],
        linkMasses=LINK_MASSES,
        linkCollisionShapeIndices=[-1, -1],
        linkVisualShapeIndices=[-1, -1],
        linkPositions=[[0, 0, 0], [l1, 0, 0]],
        linkOrientations=[[0, 0, 0, 1], [0, 0, 0, 1]],
        linkInertialFramePositions=[[l1 / 2, 0, 0], [l2 / 2, 0, 0]],
        linkInertialFrameOrientations=[[0, 0, 0, 1], [0, 0, 0, 1]],
        linkParentIndices=[0, 1],
        linkJointTypes=[p.JOINT_REVOLUTE, p.JOINT_REVOLUTE],
        linkJointAxis=[[0, 0, 1], [0, 0, 1]],
    )

    # Disable default velocity motors so we have full torque control
    for j in range(2):
        p.setJointMotorControl2(arm_id, j, p.VELOCITY_CONTROL, force=0)

    return client, arm_id


# Create and inspect
client, arm_id = create_pybullet_arm()
print(f"PyBullet arm created (client={client}, body={arm_id})")
print(f"  Joints: {p.getNumJoints(arm_id)}")
for j in range(p.getNumJoints(arm_id)):
    info = p.getJointInfo(arm_id, j)
    print(f"  Joint {j}: name={info[1].decode()}, axis={info[13]}")

# Set initial pose and verify against our FK
theta_init = np.array([np.pi / 4, -np.pi / 6])
for j in range(2):
    p.resetJointState(arm_id, j, theta_init[j])

# Read back from PyBullet
theta_pb = np.array([p.getJointState(arm_id, j)[0] for j in range(2)])
ee_np = forward_kinematics(theta_pb)
print(f"\nInitial pose: theta={theta_pb.round(3)}")
print(f"  NumPy FK -> EE = ({ee_np[0]:.4f}, {ee_np[1]:.4f})")

p.disconnect()

In [ ]:
# -- Visualization: plot the arm in 2D with matplotlib ---------------------

def plot_arm(theta, target=None, trail=None, ax=None, arm_color=COLORS["rl"],
             arm_alpha=1.0, arm_label=None, l1=L1, l2=L2, title=None):
    """Draw the 2-link arm as a top-down 2D view.

    Args:
        theta: joint angles [theta1, theta2]
        target: optional (x, y) target to mark with a star
        trail: optional Nx2 array of past EE positions
        ax: matplotlib axes (creates new if None)
        arm_color: color for arm links
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))

    t1, t2 = theta[0], theta[1]
    joint0 = np.array([0.0, 0.0])
    joint1 = np.array([l1 * np.cos(t1), l1 * np.sin(t1)])
    ee = forward_kinematics(theta, l1, l2)

    # Workspace circle (reachable boundary)
    circle = plt.Circle((0, 0), l1 + l2, fill=False, linestyle='--',
                         color=COLORS["neutral"], alpha=0.3, label='Workspace')
    ax.add_patch(circle)

    # Links
    ax.plot([joint0[0], joint1[0]], [joint0[1], joint1[1]],
            '-', color=arm_color, linewidth=6, solid_capstyle='round',
            alpha=arm_alpha)
    ax.plot([joint1[0], ee[0]], [joint1[1], ee[1]],
            '-', color=arm_color, linewidth=4, solid_capstyle='round',
            alpha=arm_alpha, label=arm_label)

    # Joints
    ax.plot(*joint0, 'ks', markersize=10, zorder=5)        # base
    ax.plot(*joint1, 'o', color=arm_color, markersize=8,
            markeredgecolor='black', zorder=5, alpha=arm_alpha)
    ax.plot(*ee, 'o', color=arm_color, markersize=10,
            markeredgecolor='black', zorder=5, alpha=arm_alpha)

    # Trail
    if trail is not None:
        trail = np.array(trail)
        ax.plot(trail[:, 0], trail[:, 1], '-', color=arm_color,
                alpha=0.3, linewidth=1.5)

    # Target
    if target is not None:
        ax.plot(target[0], target[1], '*', color=COLORS["highlight"],
                markersize=25, markeredgecolor='black', zorder=6,
                label='Target')

    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    if title:
        ax.set_title(title)
    return ax


# ── Show the arm at three different poses ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

poses = [
    (np.array([0.0, 0.0]), "Fully extended"),
    (np.array([np.pi / 4, -np.pi / 6]), "Initial pose"),
    (inverse_kinematics(np.array([0.3, 0.6])), "At target (0.3, 0.6)"),
]

for ax, (theta_show, label) in zip(axes, poses):
    plot_arm(theta_show, target=np.array([0.3, 0.6]), ax=ax, title=label)
    ax.legend(fontsize=8, loc='lower left')

plt.suptitle("2-Link Planar Arm: Three Configurations", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 17.4 Baseline: PD Controller for Reaching

Before building the AIF controller, we establish a classical control baseline. A **Proportional-Derivative (PD) controller** computes torques from the task-space position error (P term) and joint velocity (D term):

$$\boldsymbol{\tau}_\text{PD} = \mathbf{J}^\top \cdot k_p \cdot (\boldsymbol{x}^* - \boldsymbol{x}_\text{ee}) - k_d \cdot \dot{\boldsymbol{\theta}}$$

This is the bread-and-butter of classical robotics. It works well when the model is accurate and the environment is static. We will see how AIF extends this with explicit belief dynamics.

In [ ]:
# -- PD controller: classical robotics baseline ----------------------------

def pd_controller(theta, dtheta, target_xy, kp=4.0, kd=0.8):
    """Task-space PD controller using Jacobian transpose.

    Args:
        theta:     current joint angles [theta1, theta2]
        dtheta:    current joint velocities [dtheta1, dtheta2]
        target_xy: desired end-effector position [x, y]
        kp:        proportional gain (stiffness)
        kd:        derivative gain (damping)

    Returns:
        torque: joint torques [tau1, tau2]
    """
    ee = forward_kinematics(theta)
    error_task = target_xy - ee
    J = jacobian(theta)
    # Jacobian transpose maps task-space error to joint torques
    torque = J.T @ (kp * error_task) - kd * dtheta
    return torque


# ── Run the PD controller ─────────────────────────────────────────────────
theta0 = np.array([np.pi / 4, -np.pi / 6])
target_xy = np.array([0.3, 0.6])
dt = 0.01
n_steps = 600

theta = theta0.copy()
dtheta = np.zeros(2)
traj_pd, err_pd, theta_hist_pd = [], [], []

for step in range(n_steps):
    ee = forward_kinematics(theta)
    traj_pd.append(ee.copy())
    err_pd.append(np.linalg.norm(target_xy - ee))
    theta_hist_pd.append(theta.copy())

    torque = pd_controller(theta, dtheta, target_xy)
    theta, dtheta = physics_step(theta, dtheta, torque, dt)

traj_pd = np.array(traj_pd)
print(f"PD Controller: {n_steps} steps, dt={dt}, total time={n_steps*dt:.1f}s")
print(f"  Initial EE:  ({traj_pd[0, 0]:.3f}, {traj_pd[0, 1]:.3f})")
print(f"  Final EE:    ({traj_pd[-1, 0]:.3f}, {traj_pd[-1, 1]:.3f})")
print(f"  Target:      ({target_xy[0]:.3f}, {target_xy[1]:.3f})")
print(f"  Final error: {err_pd[-1]:.5f} m")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

plot_arm(theta_hist_pd[-1], target=target_xy, trail=traj_pd, ax=ax1,
         arm_color=COLORS["rl"], title="PD Controller: Final Pose")
ax1.legend(fontsize=9)

ax2.plot(np.arange(n_steps) * dt, err_pd, color=COLORS["rl"], linewidth=2)
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("End-effector error (m)")
ax2.set_title("PD Controller: Convergence")
ax2.axhline(0.01, color='gray', linestyle='--', alpha=0.5, label='1cm threshold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 17.5 Active Inference Controller for Reaching

Now we build the AIF controller. The key difference from PD control: the agent maintains **beliefs** $\boldsymbol{\mu}$ about its own joint angles, updated through perceptual inference. It does not simply react to sensor readings -- it *infers* the most likely state given noisy proprioception and its generative model.

The control loop has two interleaved processes:

1. **Perception** (fast, inner loop): update beliefs to minimize free energy
$$\dot{\boldsymbol{\mu}} = \pi_z (\boldsymbol{o} - \boldsymbol{\mu}) + \mathbf{J}(\boldsymbol{\mu})^\top \pi_w (\boldsymbol{x}^* - g(\boldsymbol{\mu}))$$

2. **Action** (slow, outer loop): apply torques based on believed state
$$\boldsymbol{\tau} = \mathbf{J}(\boldsymbol{\mu})^\top \cdot \kappa_p \cdot (\boldsymbol{x}^* - g(\boldsymbol{\mu})) - \kappa_d \cdot \dot{\boldsymbol{\theta}}$$

Notice the structural similarity to PD control -- but now the torques are computed from *beliefs*, not raw sensor data. When sensors are noisy or unreliable, this makes a profound difference.

In [ ]:
# -- Active Inference controller --------------------------------------------

class ActiveInferenceController:
    """Continuous-state Active Inference controller for a 2-link arm.

    Implements the perception-action loop:
      - Perception: gradient descent on free energy to update beliefs
      - Action: torques computed from believed state to fulfill predictions

    All math is explicit numpy -- no hidden library calls.
    """

    def __init__(self, theta_init, target_xy,
                 pi_z=100.0, pi_w=50.0,       # precisions
                 kp=4.0, kd=0.8,               # action gains
                 n_inner=5, lr_belief=0.01,     # belief update params
                 obs_noise=0.01):               # proprioceptive noise std
        # Beliefs
        self.mu = theta_init.copy()            # believed joint angles

        # Target (preferences)
        self.target_xy = target_xy.copy()

        # Precisions
        self.pi_z = pi_z    # proprioceptive precision (1/sigma_z^2)
        self.pi_w = pi_w    # goal/preference precision (1/sigma_w^2)

        # Action parameters
        self.kp = kp
        self.kd = kd

        # Perception parameters
        self.n_inner = n_inner       # inner-loop iterations per control step
        self.lr_belief = lr_belief   # belief learning rate

        # Sensor noise
        self.obs_noise = obs_noise

        # Logging
        self.fe_history = []
        self.belief_history = []

    def step(self, theta_true, dtheta_true):
        """One AIF control step: perceive, then act.

        Args:
            theta_true: actual joint angles (ground truth from physics)
            dtheta_true: actual joint velocities

        Returns:
            torque: joint torques to apply
        """
        # ── Observe (noisy proprioception) ────────────────────────────
        obs = theta_true + np.random.randn(2) * self.obs_noise

        # ── Perceptual inference (inner loop) ─────────────────────────
        # Update beliefs mu to minimize variational free energy F
        for _ in range(self.n_inner):
            # Prediction error 1: proprioceptive
            eps_z = obs - self.mu

            # Prediction error 2: goal/sensory (via generative model)
            ee_pred = forward_kinematics(self.mu)
            eps_w = self.target_xy - ee_pred

            # Free energy gradient w.r.t. beliefs
            J_mu = jacobian(self.mu)
            dF_dmu = -self.pi_z * eps_z - J_mu.T @ (self.pi_w * eps_w)

            # Gradient descent on beliefs
            self.mu = self.mu - self.lr_belief * dF_dmu

        self.belief_history.append(self.mu.copy())

        # ── Compute free energy (for logging) ─────────────────────────
        ee_mu = forward_kinematics(self.mu)
        F = (0.5 * self.pi_z * np.sum((obs - self.mu) ** 2)
             + 0.5 * self.pi_w * np.sum((self.target_xy - ee_mu) ** 2))
        self.fe_history.append(F)

        # ── Active inference: act to minimize F ───────────────────────
        # Torques from believed state (not raw sensors!)
        ee_mu = forward_kinematics(self.mu)
        error_mu = self.target_xy - ee_mu
        J_mu = jacobian(self.mu)
        torque = J_mu.T @ (self.kp * error_mu) - self.kd * dtheta_true

        return torque


# ── Run the AIF controller ────────────────────────────────────────────────
np.random.seed(42)
theta = theta0.copy()
dtheta = np.zeros(2)

aif_ctrl = ActiveInferenceController(
    theta_init=theta0,
    target_xy=target_xy,
    pi_z=100.0,   # high proprioceptive precision (trust sensors)
    pi_w=50.0,    # moderate goal precision
    kp=4.0,       # same gains as PD for fair comparison
    kd=0.8,
    obs_noise=0.01,
)

traj_aif, err_aif, theta_hist_aif = [], [], []

for step in range(n_steps):
    ee = forward_kinematics(theta)
    traj_aif.append(ee.copy())
    err_aif.append(np.linalg.norm(target_xy - ee))
    theta_hist_aif.append(theta.copy())

    torque = aif_ctrl.step(theta, dtheta)
    theta, dtheta = physics_step(theta, dtheta, torque, dt)

traj_aif = np.array(traj_aif)
print(f"AIF Controller: {n_steps} steps, dt={dt}")
print(f"  Initial EE:  ({traj_aif[0, 0]:.3f}, {traj_aif[0, 1]:.3f})")
print(f"  Final EE:    ({traj_aif[-1, 0]:.3f}, {traj_aif[-1, 1]:.3f})")
print(f"  Target:      ({target_xy[0]:.3f}, {target_xy[1]:.3f})")
print(f"  Final error: {err_aif[-1]:.5f} m")
print(f"  Final F:     {aif_ctrl.fe_history[-1]:.4f}")

## 17.6 Comparing Controllers: PD vs RL vs AIF

Now we add a simple **RL controller** (a learned linear policy trained with REINFORCE) and compare all three approaches on the reaching task. This mirrors the "Rosetta Stone" perspective from Module 13: the same task, three different frameworks.

The RL agent learns a policy $\boldsymbol{\tau} = \mathbf{W} \cdot \boldsymbol{s}$ where $\boldsymbol{s}$ includes joint angles, velocities, and the target-relative EE error. It maximizes reward $r = -\|\boldsymbol{x}^* - \boldsymbol{x}_\text{ee}\|^2$.

In [ ]:
# -- RL controller: linear policy trained with REINFORCE --------------------

class RLController:
    """Simple linear policy gradient controller for reaching.

    State: [theta1, theta2, dtheta1, dtheta2, error_x, error_y]  (6D)
    Action: [tau1, tau2]  (2D)
    Policy: tau = clip(W @ state)  (linear, 2x6 weight matrix)
    Reward: -||target - ee||^2
    """

    def __init__(self, target_xy, lr=0.01, noise_std=0.2):
        self.target_xy = target_xy.copy()
        self.W = np.zeros((2, 6))       # policy weights
        self.lr = lr
        self.noise_std = noise_std

    def _get_state(self, theta, dtheta):
        ee = forward_kinematics(theta)
        error = self.target_xy - ee
        return np.concatenate([theta, dtheta, error])

    def act(self, theta, dtheta, explore=True):
        """Select action (torque) from linear policy."""
        state = self._get_state(theta, dtheta)
        torque = self.W @ state
        if explore:
            torque = torque + np.random.randn(2) * self.noise_std
        return np.clip(torque, -5.0, 5.0)

    def train(self, n_episodes=300, n_steps_per_ep=300, dt=0.01,
              theta_init=None, verbose=True):
        """Train with REINFORCE (policy gradient)."""
        if theta_init is None:
            theta_init = np.array([np.pi / 4, -np.pi / 6])

        reward_history = []

        for ep in range(n_episodes):
            theta = theta_init.copy()
            dtheta = np.zeros(2)
            states, actions, rewards = [], [], []

            for step in range(n_steps_per_ep):
                state = self._get_state(theta, dtheta)
                torque = self.act(theta, dtheta, explore=True)
                theta, dtheta = physics_step(theta, dtheta, torque, dt)

                ee = forward_kinematics(theta)
                reward = -np.sum((self.target_xy - ee) ** 2)

                states.append(state)
                actions.append(torque)
                rewards.append(reward)

            # Compute returns (cumulative discounted reward)
            returns = np.zeros(n_steps_per_ep)
            G = 0.0
            for t in reversed(range(n_steps_per_ep)):
                G = rewards[t] + 0.99 * G
                returns[t] = G

            # Normalize returns
            ret_std = returns.std()
            if ret_std > 1e-8:
                returns = (returns - returns.mean()) / ret_std

            # REINFORCE update: dW ~ sum_t return_t * d(log pi)/dW
            # For linear Gaussian: d(log pi)/dW = (a - W@s) @ s^T / sigma^2
            for t in range(n_steps_per_ep):
                s = states[t]
                a = actions[t]
                mean_a = np.clip(self.W @ s, -5.0, 5.0)
                grad = np.outer(a - mean_a, s) / (self.noise_std ** 2)
                # Clip gradient for stability
                grad = np.clip(grad, -1.0, 1.0)
                self.W += self.lr * returns[t] * grad

            ep_reward = sum(rewards)
            reward_history.append(ep_reward)

            if verbose and (ep + 1) % 100 == 0:
                print(f"  Episode {ep+1:3d}: total reward = {ep_reward:.2f}")

        return reward_history


# ── Train the RL controller ───────────────────────────────────────────────
np.random.seed(42)
rl_ctrl = RLController(target_xy, lr=0.002, noise_std=0.2)
print("Training RL controller (REINFORCE, 300 episodes)...")
rl_rewards = rl_ctrl.train(n_episodes=300, n_steps_per_ep=300, dt=dt,
                           theta_init=theta0, verbose=True)

# ── Evaluate the RL controller (no exploration noise) ─────────────────────
theta = theta0.copy()
dtheta = np.zeros(2)
traj_rl, err_rl, theta_hist_rl = [], [], []

for step in range(n_steps):
    ee = forward_kinematics(theta)
    traj_rl.append(ee.copy())
    err_rl.append(np.linalg.norm(target_xy - ee))
    theta_hist_rl.append(theta.copy())

    torque = rl_ctrl.act(theta, dtheta, explore=False)
    theta, dtheta = physics_step(theta, dtheta, torque, dt)

traj_rl = np.array(traj_rl)
print(f"\nRL Controller (eval): final error = {err_rl[-1]:.5f} m")

In [ ]:
# -- Compare all three controllers ------------------------------------------

fig = plt.figure(figsize=(18, 10))

# ── Panel 1: Trajectories in workspace ────────────────────────────────────
ax1 = fig.add_subplot(2, 3, (1, 4))

# Workspace boundary
circle = plt.Circle((0, 0), L1 + L2, fill=False, linestyle='--',
                     color=COLORS["neutral"], alpha=0.3)
ax1.add_patch(circle)

# Target
ax1.plot(target_xy[0], target_xy[1], '*', color=COLORS["highlight"],
         markersize=25, markeredgecolor='black', zorder=6, label='Target')

# Start
ee_start = forward_kinematics(theta0)
ax1.plot(ee_start[0], ee_start[1], 'ko', markersize=10, label='Start')

# Trajectories
ax1.plot(traj_pd[:, 0], traj_pd[:, 1], '-', color=COLORS["rl"],
         linewidth=2, alpha=0.7, label=f'PD (err={err_pd[-1]:.4f})')
ax1.plot(traj_rl[:, 0], traj_rl[:, 1], '-', color=COLORS["reward"],
         linewidth=2, alpha=0.7, label=f'RL (err={err_rl[-1]:.4f})')
ax1.plot(traj_aif[:, 0], traj_aif[:, 1], '-', color=COLORS["aif"],
         linewidth=2, alpha=0.7, label=f'AIF (err={err_aif[-1]:.4f})')

ax1.set_xlim(-0.2, 1.0)
ax1.set_ylim(-0.2, 1.0)
ax1.set_aspect('equal')
ax1.set_xlabel('x (m)')
ax1.set_ylabel('y (m)')
ax1.set_title('End-Effector Trajectories')
ax1.legend(fontsize=9, loc='lower right')
ax1.grid(True, alpha=0.3)

# ── Panel 2: Error convergence ────────────────────────────────────────────
ax2 = fig.add_subplot(2, 3, 2)
time = np.arange(n_steps) * dt
ax2.plot(time, err_pd, color=COLORS["rl"], linewidth=2, label='PD')
ax2.plot(time, err_rl, color=COLORS["reward"], linewidth=2, label='RL')
ax2.plot(time, err_aif, color=COLORS["aif"], linewidth=2, label='AIF')
ax2.axhline(0.05, color='gray', linestyle='--', alpha=0.5, label='5cm')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('EE Error (m)')
ax2.set_title('Convergence')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# ── Panel 3: RL training curve ────────────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)
window = 20
smoothed = np.convolve(rl_rewards, np.ones(window) / window, mode='valid')
ax3.plot(smoothed, color=COLORS["reward"], linewidth=2)
ax3.set_xlabel('Episode')
ax3.set_ylabel('Total Reward')
ax3.set_title('RL Training (REINFORCE)')
ax3.grid(True, alpha=0.3)

# ── Panel 4: Free energy over time (AIF only) ────────────────────────────
ax4 = fig.add_subplot(2, 3, 5)
ax4.plot(time, aif_ctrl.fe_history, color=COLORS["aif"], linewidth=2)
ax4.set_xlabel('Time (s)')
ax4.set_ylabel('Free Energy F')
ax4.set_title('AIF: Free Energy Minimization')
ax4.grid(True, alpha=0.3)

# ── Panel 5: Belief vs truth (AIF only) ──────────────────────────────────
ax5 = fig.add_subplot(2, 3, 6)
beliefs = np.array(aif_ctrl.belief_history)
thetas = np.array(theta_hist_aif)
ax5.plot(time, thetas[:, 0], '-', color=COLORS["neutral"], linewidth=2,
         label=r'$\theta_1$ (true)')
ax5.plot(time, beliefs[:, 0], '--', color=COLORS["aif"], linewidth=2,
         label=r'$\mu_1$ (belief)')
ax5.plot(time, thetas[:, 1], '-', color=COLORS["neutral"], linewidth=1.5,
         alpha=0.6, label=r'$\theta_2$ (true)')
ax5.plot(time, beliefs[:, 1], '--', color=COLORS["aif"], linewidth=1.5,
         alpha=0.6, label=r'$\mu_2$ (belief)')
ax5.set_xlabel('Time (s)')
ax5.set_ylabel('Angle (rad)')
ax5.set_title('AIF: Beliefs Track True State')
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.3)

plt.suptitle('Three Controllers for Embodied Reaching', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nSummary:")
print(f"  PD:  final error = {err_pd[-1]:.5f} m  (no training required)")
print(f"  RL:  final error = {err_rl[-1]:.5f} m  (300 episodes training)")
print(f"  AIF: final error = {err_aif[-1]:.5f} m  (no training, uses generative model)")

In [ ]:
# -- Dual Perspective: Embodied Control ------------------------------------

dual_perspective_box(
    rl_text=(
        "<b>The RL agent</b> learns a policy &pi;(a|s) that maps sensor readings "
        "to torques, maximizing cumulative reward r = -||target - EE||<sup>2</sup>. "
        "It needs hundreds of training episodes to discover the mapping from joint "
        "angles to effective torques. The trained policy is a black box -- it works "
        "but does not explain <em>why</em> a particular torque is chosen. If the "
        "target moves or the arm's dynamics change, the policy must be retrained."
    ),
    aif_text=(
        "<b>The AIF agent</b> has a generative model (forward kinematics + dynamics) "
        "and preferences (target position). It maintains beliefs about joint angles "
        "and updates them via prediction error minimization. Action is not reward "
        "maximization -- it is <em>making predictions come true</em>. The Jacobian "
        "transpose naturally maps sensory prediction errors to motor commands. "
        "No training is needed: the generative model encodes the physics. If the "
        "target moves, the agent adapts immediately through online belief updating."
    ),
    title="Embodied Reaching: RL vs Active Inference",
)

## 17.7 Perturbation Robustness

The real advantage of Active Inference over reactive controllers emerges under perturbation. We test three disruptions:

1. **Target jump**: the target moves mid-reach (at t=2s)
2. **Noisy sensors**: proprioceptive noise increases 10x
3. **Morphology change**: the second link becomes 50% longer (simulating a tool grasp)

The PD controller reacts directly to sensor readings and is vulnerable to noise. The RL controller has a fixed policy and cannot adapt to changes it was not trained on. The AIF controller adapts online through belief updating -- its generative model filters noise and its prediction errors automatically adjust to new conditions.

In [ ]:
# -- Perturbation experiments -----------------------------------------------

def run_perturbation_experiment(perturbation, n_steps=800, dt=0.01):
    """Run PD, RL, and AIF under a specified perturbation.

    Args:
        perturbation: one of 'target_jump', 'noisy_sensors', 'morphology_change'

    Returns:
        dict with error histories for each controller
    """
    target1 = np.array([0.3, 0.6])
    target2 = np.array([0.6, 0.3])
    switch_step = 300   # perturbation at t = 3s
    obs_noise_base = 0.01
    obs_noise_high = 0.1

    results = {}
    for name in ['PD', 'RL', 'AIF']:
        np.random.seed(42)
        theta = theta0.copy()
        dtheta = np.zeros(2)
        errors = []

        if name == 'AIF':
            ctrl = ActiveInferenceController(
                theta_init=theta0, target_xy=target1,
                pi_z=100.0, pi_w=50.0, kp=4.0, kd=0.8,
                obs_noise=obs_noise_base,
            )

        for step in range(n_steps):
            # Determine current target
            if perturbation == 'target_jump' and step >= switch_step:
                current_target = target2
            else:
                current_target = target1

            # Determine current noise level
            if perturbation == 'noisy_sensors' and step >= switch_step:
                current_noise = obs_noise_high
            else:
                current_noise = obs_noise_base

            # Determine FK parameters (for morphology change)
            if perturbation == 'morphology_change' and step >= switch_step:
                l2_eff = L2 * 1.5   # link grows 50%
            else:
                l2_eff = L2

            ee = forward_kinematics(theta, L1, l2_eff)
            errors.append(np.linalg.norm(current_target - ee))

            if name == 'PD':
                # PD uses true (possibly noisy) readings
                obs_theta = theta + np.random.randn(2) * current_noise
                ee_obs = forward_kinematics(obs_theta)
                error_task = current_target - ee_obs
                J = jacobian(obs_theta)
                torque = J.T @ (4.0 * error_task) - 0.8 * dtheta

            elif name == 'RL':
                torque = rl_ctrl.act(theta, dtheta, explore=False)
                # RL was trained for target1, no adaptation

            elif name == 'AIF':
                ctrl.target_xy = current_target.copy()
                ctrl.obs_noise = current_noise
                torque = ctrl.step(theta, dtheta)

            theta, dtheta = physics_step(theta, dtheta, torque, dt)

        results[name] = errors

    return results


# ── Run all three perturbation experiments ─────────────────────────────────
perturbations = {
    'target_jump': 'Target Jump at t=3s',
    'noisy_sensors': '10x Sensor Noise at t=3s',
    'morphology_change': 'Link Length +50% at t=3s',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (perturb_key, perturb_title) in zip(axes, perturbations.items()):
    results = run_perturbation_experiment(perturb_key)
    n_perturb = len(results['PD'])
    time_p = np.arange(n_perturb) * 0.01

    ax.plot(time_p, results['PD'], color=COLORS["rl"], linewidth=2,
            label='PD', alpha=0.8)
    ax.plot(time_p, results['RL'], color=COLORS["reward"], linewidth=2,
            label='RL', alpha=0.8)
    ax.plot(time_p, results['AIF'], color=COLORS["aif"], linewidth=2,
            label='AIF', alpha=0.8)

    ax.axvline(3.0, color='red', linestyle='--', alpha=0.4, label='Perturbation')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('EE Error (m)')
    ax.set_title(perturb_title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)

    # Print final errors
    print(f"\n{perturb_title}:")
    for name in ['PD', 'RL', 'AIF']:
        print(f"  {name}: final error = {results[name][-1]:.4f} m")

plt.suptitle('Perturbation Robustness: PD vs RL vs AIF', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 17.8 Precision-Weighted Prediction Errors: The AIF Advantage

The AIF controller has a unique mechanism that neither PD nor RL possess: **precision weighting**. By adjusting $\pi_z$ (proprioceptive precision) and $\pi_w$ (goal precision), the agent can flexibly balance:

- **Trust in sensors vs. trust in goals**: high $\pi_z$ means "believe what I feel"; high $\pi_w$ means "do what I want"
- **Compliance vs. stiffness**: low $\pi_w$ makes the arm compliant (soft, yielding to perturbation); high $\pi_w$ makes it stiff (insistent on reaching the target)

This is directly analogous to the precision dynamics in Module 15 (hierarchical AIF) and the gamma parameter in Module 16 (multi-agent commons). Precision is the FEP's universal attention mechanism.

In [ ]:
# -- Precision sweep: how pi_z and pi_w shape behavior ---------------------

precision_configs = [
    (10.0,   50.0, "Low sensor trust,\nhigh goal drive"),
    (100.0,  50.0, "High sensor trust,\nhigh goal drive (default)"),
    (100.0,  10.0, "High sensor trust,\nlow goal drive (compliant)"),
    (100.0, 200.0, "High sensor trust,\nvery high goal drive (stiff)"),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (pi_z, pi_w, label) in zip(axes, precision_configs):
    np.random.seed(42)
    theta = theta0.copy()
    dtheta = np.zeros(2)
    ctrl = ActiveInferenceController(
        theta_init=theta0, target_xy=target_xy,
        pi_z=pi_z, pi_w=pi_w, kp=4.0, kd=0.8,
        obs_noise=0.05,  # moderate noise to show filtering effect
    )

    traj = []
    for step in range(n_steps):
        traj.append(forward_kinematics(theta).copy())
        torque = ctrl.step(theta, dtheta)
        theta, dtheta = physics_step(theta, dtheta, torque, dt)

    traj = np.array(traj)

    # Plot trajectory
    circle = plt.Circle((0, 0), L1 + L2, fill=False, linestyle='--',
                         color=COLORS["neutral"], alpha=0.3)
    ax.add_patch(circle)
    ax.plot(traj[:, 0], traj[:, 1], '-', color=COLORS["aif"],
            linewidth=1.5, alpha=0.7)
    ax.plot(target_xy[0], target_xy[1], '*', color=COLORS["highlight"],
            markersize=20, markeredgecolor='black', zorder=5)
    ax.plot(traj[0, 0], traj[0, 1], 'ko', markersize=8)

    final_err = np.linalg.norm(target_xy - traj[-1])
    ax.set_title(f"{label}\n$\\pi_z$={pi_z}, $\\pi_w$={pi_w}\nerr={final_err:.4f}",
                 fontsize=10)
    ax.set_xlim(-0.2, 1.0)
    ax.set_ylim(-0.2, 1.0)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.suptitle('Precision Weighting Shapes Behavior', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("Observation: precision acts like a tuning dial between compliance and stiffness.")
print("In biological systems, this maps to attention and motor gain modulation.")

## 17.9 PyBullet Integration: Physics-Engine Validation

So far we used our simplified dynamics model (`physics_step`) for clarity. Now let us verify that the AIF controller also works with PyBullet's full rigid-body physics, which includes proper inertia computation, contact dynamics, and numerical integration. This demonstrates that AIF is not tied to a particular dynamics model -- it works with any physics engine as long as it can observe joint states and apply torques.

In [ ]:
# -- Run AIF controller with PyBullet physics engine -----------------------

# Create the arm in PyBullet
client, arm_id = create_pybullet_arm()

# Set initial pose
for j in range(2):
    p.resetJointState(arm_id, j, theta0[j])

# Create AIF controller
np.random.seed(42)
aif_pb = ActiveInferenceController(
    theta_init=theta0, target_xy=target_xy,
    pi_z=100.0, pi_w=50.0, kp=5.0, kd=1.0,  # higher gains for PyBullet dynamics
    obs_noise=0.01,
)

pb_steps_per_ctrl = 10   # physics substeps per control step
n_ctrl_steps = 400
pb_traj, pb_err = [], []

for step in range(n_ctrl_steps):
    # Read joint states from PyBullet
    theta_pb = np.array([p.getJointState(arm_id, j)[0] for j in range(2)])
    dtheta_pb = np.array([p.getJointState(arm_id, j)[1] for j in range(2)])

    ee = forward_kinematics(theta_pb)
    pb_traj.append(ee.copy())
    pb_err.append(np.linalg.norm(target_xy - ee))

    # AIF controller
    torque = aif_pb.step(theta_pb, dtheta_pb)

    # Apply torques and step physics
    for j in range(2):
        p.setJointMotorControl2(arm_id, j, p.TORQUE_CONTROL,
                                force=float(torque[j]))
    for _ in range(pb_steps_per_ctrl):
        p.stepSimulation()

p.disconnect()

pb_traj = np.array(pb_traj)
print(f"PyBullet + AIF: {n_ctrl_steps} control steps")
print(f"  Initial EE: ({pb_traj[0, 0]:.3f}, {pb_traj[0, 1]:.3f})")
print(f"  Final EE:   ({pb_traj[-1, 0]:.3f}, {pb_traj[-1, 1]:.3f})")
print(f"  Target:     ({target_xy[0]:.3f}, {target_xy[1]:.3f})")
print(f"  Final error: {pb_err[-1]:.5f} m")

# ── Compare trajectories: simplified physics vs PyBullet ──────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Trajectory comparison
circle = plt.Circle((0, 0), L1 + L2, fill=False, linestyle='--',
                     color=COLORS["neutral"], alpha=0.3)
ax1.add_patch(circle)
ax1.plot(traj_aif[:, 0], traj_aif[:, 1], '-', color=COLORS["aif"],
         linewidth=2, alpha=0.7, label='Simplified physics')
ax1.plot(pb_traj[:, 0], pb_traj[:, 1], '--', color=COLORS["epistemic"],
         linewidth=2, alpha=0.7, label='PyBullet physics')
ax1.plot(target_xy[0], target_xy[1], '*', color=COLORS["highlight"],
         markersize=25, markeredgecolor='black', zorder=6)
ax1.set_xlim(-0.2, 1.0)
ax1.set_ylim(-0.2, 1.0)
ax1.set_aspect('equal')
ax1.set_title('EE Trajectories: Simplified vs PyBullet')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Error comparison
ax2.plot(err_aif, color=COLORS["aif"], linewidth=2, label='Simplified', alpha=0.8)
ax2.plot(pb_err, color=COLORS["epistemic"], linewidth=2, label='PyBullet', alpha=0.8)
ax2.set_xlabel('Control step')
ax2.set_ylabel('EE Error (m)')
ax2.set_title('Convergence Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('AIF Works With Different Physics Backends', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# -- Dual Perspective: Perturbation Robustness -----------------------------

dual_perspective_box(
    rl_text=(
        "<b>Robustness in RL</b> requires either (a) domain randomization during "
        "training (expose the policy to many perturbations so it generalizes), or "
        "(b) online adaptation methods like meta-RL. The standard trained policy is "
        "brittle: it encodes a fixed mapping from states to actions. When the "
        "environment changes (new target, new dynamics), the mapping is wrong and "
        "must be re-learned. Sim-to-real transfer in robotics is hard precisely "
        "because the RL policy overfits to the simulator's dynamics."
    ),
    aif_text=(
        "<b>Robustness in AIF</b> is built-in through the perception-action loop. "
        "When the target moves, the sensory prediction error immediately changes, "
        "driving new torques. When sensors become noisy, the precision weighting "
        "automatically down-weights unreliable channels. When the body changes, "
        "the proprioceptive prediction errors signal model mismatch, prompting "
        "belief revision. The generative model does not need retraining -- it "
        "adapts online through the same free energy minimization that drives "
        "normal reaching."
    ),
    title="Why AIF is Naturally Robust to Perturbation",
)

## 17.10 Exercises

### Exercise 1 (Guided): Add a Third Joint

Extend the arm to 3 links. You will need to:
- Update `forward_kinematics` to handle 3 joint angles
- Update `jacobian` to return a 2x3 matrix
- Update `physics_step` for 3 DOF
- Create a new `ActiveInferenceController` with 3D beliefs

**Key question**: How does the AIF agent handle the *redundancy* (3 joints for a 2D target means infinitely many solutions)? Observe that the Jacobian transpose naturally selects a solution that minimizes joint motion.

### Exercise 2 (Intermediate): Obstacle Avoidance

Add a circular obstacle in the workspace. Modify the AIF free energy to include a *repulsive* prediction error:

$$F_\text{obs} = \frac{1}{2} \pi_\text{obs} \cdot \max(0, r_\text{obs} - \|g(\boldsymbol{\mu}) - \boldsymbol{x}_\text{obs}\|)^2$$

where $r_\text{obs}$ is the obstacle radius and $\boldsymbol{x}_\text{obs}$ is its center. This penalizes beliefs where the EE is inside the obstacle.

### Exercise 3 (Open-ended): Evolving Morphology

Inspired by Bongard's evolutionary robotics: run the AIF controller with different arm morphologies (link lengths $L_1, L_2$). Sweep a grid of morphologies and measure reaching accuracy. Which body plans are easiest to control? Does the AIF controller's performance depend on the condition number of the Jacobian (a measure of kinematic manipulability)?

In [ ]:
# -- Exercise 1 starter: 3-link arm ----------------------------------------

def forward_kinematics_3(theta, l1=0.4, l2=0.3, l3=0.2):
    """FK for a 3-link planar arm. Exercise: implement this."""
    t1, t2, t3 = theta[0], theta[1], theta[2]
    x = (l1 * np.cos(t1)
         + l2 * np.cos(t1 + t2)
         + l3 * np.cos(t1 + t2 + t3))
    y = (l1 * np.sin(t1)
         + l2 * np.sin(t1 + t2)
         + l3 * np.sin(t1 + t2 + t3))
    return np.array([x, y])


def jacobian_3(theta, l1=0.4, l2=0.3, l3=0.2):
    """Jacobian for 3-link arm (2x3). Exercise: implement this."""
    t1, t2, t3 = theta[0], theta[1], theta[2]
    s1, c1 = np.sin(t1), np.cos(t1)
    s12, c12 = np.sin(t1 + t2), np.cos(t1 + t2)
    s123, c123 = np.sin(t1 + t2 + t3), np.cos(t1 + t2 + t3)
    return np.array([
        [-l1*s1 - l2*s12 - l3*s123, -l2*s12 - l3*s123, -l3*s123],
        [ l1*c1 + l2*c12 + l3*c123,  l2*c12 + l3*c123,  l3*c123],
    ])


# Quick test
theta3_test = np.array([np.pi/4, np.pi/6, -np.pi/4])
ee3 = forward_kinematics_3(theta3_test)
J3 = jacobian_3(theta3_test)
print(f"3-link FK: theta={theta3_test.round(2)} -> EE=({ee3[0]:.3f}, {ee3[1]:.3f})")
print(f"3-link Jacobian shape: {J3.shape}")
print(f"Jacobian condition number: {np.linalg.cond(J3 @ J3.T):.1f}")
print("\nExercise: create a 3-joint AIF controller and run the reaching task.")
print("Observe how the redundant DOF is resolved by the Jacobian transpose.")

In [ ]:
# -- Exercise 3 starter: morphology sweep ----------------------------------

def evaluate_morphology(l1, l2, target_xy, n_steps=300, dt=0.01):
    """Run AIF reaching with given link lengths, return final error."""
    # Check reachability
    if np.linalg.norm(target_xy) > (l1 + l2) * 0.95:
        return np.nan  # unreachable

    np.random.seed(42)
    theta = np.array([np.pi / 4, -np.pi / 6])
    dtheta = np.zeros(2)

    ctrl = ActiveInferenceController(
        theta_init=theta, target_xy=target_xy,
        pi_z=100.0, pi_w=50.0, kp=2.0, kd=0.5, obs_noise=0.01,
    )
    # Override FK and Jacobian closures to use custom link lengths
    ctrl._fk = lambda th: forward_kinematics(th, l1, l2)
    ctrl._jac = lambda th: jacobian(th, l1, l2)

    for step in range(n_steps):
        torque = ctrl.step(theta, dtheta)
        theta, dtheta = physics_step(theta, dtheta, torque, dt)

    ee = forward_kinematics(theta, l1, l2)
    return np.linalg.norm(target_xy - ee)


# Sweep morphologies
target_sweep = np.array([0.3, 0.5])
l1_range = np.linspace(0.2, 0.8, 10)
l2_range = np.linspace(0.1, 0.6, 10)
error_grid = np.full((len(l1_range), len(l2_range)), np.nan)

for i, l1_val in enumerate(l1_range):
    for j, l2_val in enumerate(l2_range):
        error_grid[i, j] = evaluate_morphology(l1_val, l2_val, target_sweep)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(error_grid, origin='lower', cmap='RdYlGn_r', aspect='auto',
               extent=[l2_range[0], l2_range[-1], l1_range[0], l1_range[-1]])
plt.colorbar(im, ax=ax, label='Final EE Error (m)')
ax.set_xlabel('L2 (m)')
ax.set_ylabel('L1 (m)')
ax.set_title(f'Morphology Sweep: AIF Reaching Error\n(target = {target_sweep})')
ax.grid(True, alpha=0.2)

# Mark the default morphology
ax.plot(L2, L1, 'w*', markersize=15, markeredgecolor='black', label='Default')
ax.legend()

plt.tight_layout()
plt.show()

print("Exercise: which morphologies produce the lowest error?")
print("Investigate the relationship between Jacobian condition number and performance.")

## 17.11 From Thorndike's Cats to Embodied AIF: The Full Arc

We began this curriculum with cats scratching at latches and pigeons pecking at keys. Over 17 modules we built up a unified understanding:

| Module | Key Insight |
|--------|-------------|
| 1-4 | Behavior can be understood through **value** and **prediction error** |
| 5-7 | RL maximizes reward; AIF minimizes surprise -- different objectives, same math |
| 8-10 | The Free Energy Principle derives perception, action, and planning from one imperative |
| 11-12 | Generative models can be learned from data (discrete and deep) |
| 13 | The Rosetta Stone: RL and AIF are formally equivalent |
| 14-15 | JAX scaling and hierarchical models enable real-world complexity |
| 16 | Multi-agent AIF governs shared resources (commons) |
| **17** | **Embodied AIF closes the loop: continuous bodies, physics, proprioception** |

The cat in Thorndike's puzzle box was always an embodied agent. Its learning was not abstract symbol manipulation -- it was coordinated motion of joints, muscles, and tendons, driven by prediction errors between expected and actual sensory feedback. Active Inference was the right framework all along. We just needed 16 modules to see it clearly.

> *"The brain is not a computer that happens to be attached to a body. The brain, body, and environment are a single coupled dynamical system."*
> -- Randall Beer, paraphrasing the embodied cognition tradition

---

## Beyond PyBullet: The JAX-Native Physics Frontier

This module uses PyBullet for accessibility, but the future of embodied AIF is **JAX-native physics** — keeping everything (simulation, inference, learning) on the GPU with no CPU-GPU transfer bottleneck.

**[MuJoCo XLA (MJX)](https://mujoco.readthedocs.io/en/stable/mjx.html)** — Google's JAX port of the MuJoCo physics engine. Because both physics and neural network training live in JAX, you can `vmap` over hundreds of bodies simultaneously, achieving millions of simulation steps per second. This is the natural next step from our Module 14 (JAX scaling): `vmap` over agents AND bodies.

**[MIMIC-MJX and the Virtual Rodent](https://github.com/google-deepmind/mujoco_playground)** (DeepMind) — A biomechanically accurate rat model trained via RL in MJX. The rodent learns locomotion, turning, and obstacle avoidance — all on GPU. This is the state of the art for embodied AIF/RL: a biologically plausible morphology in a differentiable physics engine, trainable end-to-end.

**[pyrosim](https://github.com/mec-lab/pyrosim)** (Bongard, UVM) — The simulator behind [Ludobots](https://www.reddit.com/r/ludobots/), built on Open Dynamics Engine (ODE). Students evolve arbitrary body plans with neural controllers. Less performant than MJX but designed for evolutionary exploration of morphology space.

**[RatInABox](https://github.com/TomGeorge1234/RatInABox)** — Lightweight spatial navigation simulator with community JAX wrappers. Pairs naturally with our neuro-nav environments (Modules 1-7) and could bridge to embodied hippocampal models.

The roadmap: PyBullet (this module, accessible) -> MJX (JAX-native, scalable) -> MIMIC-MJX (biologically realistic morphology). An Active Inference agent in MJX, controlling a virtual rodent that plans via EFE minimization — that is where Modules 1-17 lead.

---

## Further Reading

1. **Friston, K. (2010).** *The free-energy principle: a unified brain theory?* Nature Reviews Neuroscience, 11(2), 127-138. The landmark paper that framed perception, action, and learning as free energy minimization. Our embodied controller is a direct implementation of the "action as inference" idea from Section 3.

2. **Pfeifer, R. & Bongard, J. (2006).** *How the Body Shapes the Way We Think: A New View of Intelligence.* MIT Press. The foundational text on embodied intelligence. Chapter 4 on "cheap design" explains why simple bodies with good controllers outperform complex brains with bad bodies — directly relevant to our morphology sweep (Exercise 3).

3. **Lanillos, P., Meo, C., Pico, G., et al. (2021).** *Active Inference in Robotics and Artificial Agents: Survey and Challenges.* arXiv:2112.01871. The most comprehensive survey of AIF applied to real robots. Covers proprioceptive inference (Section 3), reaching (Section 4), and the relationship to classical control (Section 6) — exactly the topics of this module.

4. **Bongard, J., Levin, M., & Livingston, K. (2021).** *Living Things Are Not (20th Century) Machines: Updating Mechanism Metaphors in Light of the Modern Science of Machine Behavior.* Frontiers in Ecology and Evolution. A provocative argument that biological agents are fundamentally different from engineered machines, with implications for how we model embodied intelligence.

5. **Tschantz, A., Millidge, B., Seth, A.K., & Buckley, C.L. (2020).** *Reinforcement Learning through Active Inference.* arXiv:2002.12636. The formal proof that AIF with the right generative model recovers RL as a special case. Our reaching comparison (Section 17.6) is a concrete demonstration of this equivalence in the continuous domain.

6. **Merel, J. et al. (2020).** *Deep neuroethology of a virtual rodent.* ICLR. The DeepMind virtual rodent — a biomechanically realistic rat model trained in MuJoCo. The MIMIC-MJX framework now makes this trainable end-to-end in JAX.

7. **George, T.M. et al. (2024).** *RatInABox, a toolkit for modelling locomotion and neuronal activity in continuous environments.* eLife. The [vector cell demo](https://github.com/TomGeorge1234/RatInABox/blob/main/demos/vector_cell_demo.ipynb) implements egocentric boundary and object vector cells — the sensory representations an embodied agent uses to perceive its environment. The [path integration demo](https://github.com/TomGeorge1234/RatInABox/blob/main/demos/path_integration_example.ipynb) shows ring attractor networks for spatial tracking from velocity input alone.